# Demo — BPR + CLIP Hybrid Recommender

Loads a trained BPRClipHybrid model and displays a user's interaction history,
top-K recommendations, and held-out test items. The model blends collaborative
filtering (BPR) with multimodal content signals (CLIP text + image embeddings)
via a per‑user learned alpha weight.

In [1]:
import numpy as np

np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

import logging
logging.getLogger().handlers.clear()
logging.getLogger('httpx').setLevel(logging.WARNING)

In [2]:
from typing import Any
import torch
import torch.nn as nn
import os
import glob
import pandas as pd
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.abstract_recommender import GeneralRecommender
from recbole.model.loss import BPRLoss
from recbole.model.init import xavier_normal_initialization
from recbole.utils import init_seed, init_logger, InputType, ModelType
from IPython.display import display
from sentence_transformers import SentenceTransformer

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATASET_NAME: str = "clothing-beauty"
DATA_DIR: str = "data"
TRAIN_DIR: str = "train"
CLIP_TEXT_PATH: str = f"{DATA_DIR}/{DATASET_NAME}/clip_text_embeddings.pt"
CLIP_IMAGE_PATH: str = f"{DATA_DIR}/{DATASET_NAME}/clip_image_embeddings.pt"
SEED = 67
DEVICE = "mps"
EMBEDDING_SIZE = 64
BEAUTY_WEIGHT = 10
TOPK = 20

## 1. Create dataset

Uses the same RecBole atomic files and config as the training notebook.

In [4]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "title", "store", "price", "cold", "target"],
    },
    "embedding_size": EMBEDDING_SIZE,
    "epochs": 1,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metric_decimal_place": 6,
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
}

config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

n_users = dataset.user_num
n_items = dataset.item_num
print(f"Dataset: {n_users:,} users, {n_items:,} items")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work becaus

Dataset: 156,488 users, 352,858 items


## 2. Build CLIP multimodal embeddings

Loads precomputed CLIP text + image embeddings and fuses them (average, L2-normalized). Same logic as the training notebook.

In [5]:
logger = logging.getLogger()

title_tokens: torch.Tensor = dataset.item_feat["title"]
id2token: dict[int, str] = {v: k for k, v in dataset.field2token_id["title"].items()}
titles: list[str] = [id2token.get(tok.item(), "") for tok in title_tokens]

In [6]:
if not os.path.exists(CLIP_TEXT_PATH):
    logger.info("Encoding %d item titles with CLIP text tower...", len(titles))
    clip_model = SentenceTransformer("clip-ViT-B-32", device=str(config["device"]))
    clip_text_embs = clip_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
    clip_text_embs = torch.from_numpy(clip_text_embs).float()
    torch.save(clip_text_embs, CLIP_TEXT_PATH)
else:
    logger.info("CLIP text embeddings already exist at %s, loading cached.", CLIP_TEXT_PATH)

clip_text_embs = torch.load(CLIP_TEXT_PATH, map_location="cpu")

24 Jun 14:14    INFO  CLIP text embeddings already exist at data/clothing-beauty/clip_text_embeddings.pt, loading cached.
/var/folders/vm/77wrgjgj5wzbyghx353b7gym0000gn/T/ipykernel_85931/3746307001.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please o

In [7]:
clip_data = torch.load(CLIP_IMAGE_PATH, map_location="cpu")
clip_raw = clip_data["embeddings"]
iid_to_idx = clip_data["iid_to_idx"]

fused = clip_text_embs.clone()
n_with_image = 0

for internal_id in range(1, n_items):
    token = dataset.id2token(dataset.iid_field, internal_id)
    try:
        iid = int(token)
    except ValueError:
        continue
    dense_idx = iid_to_idx.get(iid)
    if dense_idx is None:
        continue
    img_vec = clip_raw[dense_idx]
    if img_vec.abs().sum() == 0:
        continue
    fused[internal_id] = (clip_text_embs[internal_id] + img_vec) / 2.0
    n_with_image += 1

norms = fused.norm(dim=1, keepdim=True)
norms[norms == 0] = 1.0
fused = fused / norms
fused[0] = 0.0

print(f"Items with both text+image: {n_with_image:,} / {n_items - 1:,} ({100 * n_with_image / max(n_items - 1, 1):.2f}%)")
item_multimodal_embs = fused

/var/folders/vm/77wrgjgj5wzbyghx353b7gym0000gn/T/ipykernel_85931/2215648754.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_data = torch.load(CLIP_IMAGE_PATH, map_l

Items with both text+image: 352,818 / 352,857 (99.99%)


## 3. Define BPRClipHybrid

Trainable user/item embeddings + per‑user alpha blend weight + frozen CLIP content signal.

In [8]:
class BPRClipHybrid(GeneralRecommender):
    input_type = InputType.PAIRWISE
    type = ModelType.GENERAL

    def __init__(self, config, dataset, item_content_embeddings: torch.Tensor):
        super().__init__(config, dataset)

        self.embedding_size = config["embedding_size"]

        self.user_embedding = nn.Embedding(self.n_users, self.embedding_size)
        self.item_embedding = nn.Embedding(self.n_items, self.embedding_size)
        self.loss = BPRLoss()
        self.apply(xavier_normal_initialization)

        content_dim = item_content_embeddings.shape[1]
        g = torch.Generator().manual_seed(config["seed"])
        projection = torch.randn(content_dim, self.embedding_size, generator=g) / (content_dim ** 0.5)
        item_init = item_content_embeddings.float() @ projection.to(item_content_embeddings.device)
        with torch.no_grad():
            self.item_embedding.weight.data.copy_(item_init)
        self.item_embedding.weight.data[0] = 0.0

        self.user_alpha = nn.Embedding(self.n_users, 1)
        nn.init.constant_(self.user_alpha.weight, 0.0)

        self.register_buffer("item_content_embeddings", item_content_embeddings.float())

        target = (dataset.item_feat["target"] == 1.0).float()
        self.register_buffer("item_target", target)

        train_inter = dataset.inter_feat
        users = train_inter[self.USER_ID]
        items = train_inter[self.ITEM_ID]
        item_weight = 1.0 + (BEAUTY_WEIGHT - 1.0) * self.item_target[items]
        profile_sum = torch.zeros(self.n_users, content_dim)
        profile_cnt = torch.zeros(self.n_users, 1)
        profile_sum.index_add_(0, users, item_weight.unsqueeze(-1) * self.item_content_embeddings[items])
        profile_cnt.index_add_(0, users, item_weight.unsqueeze(-1))
        self.register_buffer("user_content_profile", profile_sum / profile_cnt.clamp(min=1))

    @staticmethod
    def _row_normalize(x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return (x - mean) / (std + 1e-8)

    def calculate_loss(self, interaction):
        user = interaction[self.USER_ID]
        pos_item = interaction[self.ITEM_ID]
        neg_item = interaction[self.NEG_ITEM_ID]

        user_e = self.user_embedding(user)
        pos_e = self.item_embedding(pos_item)
        neg_e = self.item_embedding(neg_item)

        alpha = torch.sigmoid(self.user_alpha(user)).squeeze(-1)

        bpr_pos = (user_e * pos_e).sum(dim=-1)
        bpr_neg = (user_e * neg_e).sum(dim=-1)

        content_pos = (self.user_content_profile[user] * self.item_content_embeddings[pos_item]).sum(dim=-1)
        content_neg = (self.user_content_profile[user] * self.item_content_embeddings[neg_item]).sum(dim=-1)

        pos_score = alpha * bpr_pos + (1 - alpha) * content_pos
        neg_score = alpha * bpr_neg + (1 - alpha) * content_neg
        pos_target = self.item_target[pos_item]
        weights = torch.where(pos_target == 1.0, BEAUTY_WEIGHT, 1.0)
        raw_loss = -torch.log(1e-10 + torch.sigmoid(pos_score - neg_score))
        return (raw_loss * weights).mean()

    def predict(self, interaction):
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]

        bpr_score = (self.user_embedding(user) * self.item_embedding(item)).sum(dim=-1)
        content_score = (self.user_content_profile[user] * self.item_content_embeddings[item]).sum(dim=-1)
        alpha = torch.sigmoid(self.user_alpha(user)).squeeze(-1)
        return alpha * bpr_score + (1 - alpha) * content_score

    def full_sort_predict(self, interaction):
        user = interaction[self.USER_ID]

        bpr_scores = torch.matmul(self.user_embedding(user), self.item_embedding.weight.t())
        content_scores = torch.matmul(self.user_content_profile[user], self.item_content_embeddings.t())

        alpha = torch.sigmoid(self.user_alpha(user))
        blended = alpha * self._row_normalize(bpr_scores) + (1 - alpha) * self._row_normalize(content_scores)
        return blended.view(-1)

## 4. Load checkpoint

Auto‑picks the latest BPR checkpoint that contains BPRClipHybrid keys (not standard BPR).

In [9]:
def find_best_hybrid_checkpoint(saved_dir: str, expected_n_items: int) -> str:
    paths = sorted(glob.glob(os.path.join(saved_dir, "BPR-*.pth")), key=os.path.getmtime, reverse=True)
    hybrid_keys = {"item_content_embeddings", "user_content_profile", "user_alpha.weight"}
    for p in paths:
        ckpt = torch.load(p, map_location="cpu", weights_only=False)
        sd = ckpt["state_dict"]
        sd_keys = set(sd.keys())
        if hybrid_keys.issubset(sd_keys) and sd["item_embedding.weight"].shape[0] == expected_n_items:
            return p
    msg = f"No BPRClipHybrid checkpoint with {expected_n_items} items found in {saved_dir}"
    raise FileNotFoundError(msg)

checkpoint_path = find_best_hybrid_checkpoint(os.path.join(TRAIN_DIR, "saved"), n_items)
print(f"Checkpoint: {checkpoint_path}")

Checkpoint: train/saved/BPR-Jun-23-2026_21-11-56.pth


In [10]:
model = BPRClipHybrid(config, train_data.dataset, item_multimodal_embs).to(DEVICE)

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
missing, unexpected = model.load_state_dict(checkpoint["state_dict"], strict=False)
print(f"Missing keys: {missing}")
print(f"Unexpected keys: {unexpected}")
print(f"Loaded from epoch {checkpoint['epoch']} (best valid NDCG@20 = {checkpoint['best_valid_score']:.6f})")

model.eval()

Missing keys: []
Unexpected keys: []
Loaded from epoch 11 (best valid NDCG@20 = 0.010179)


BPRClipHybrid(
  (user_embedding): Embedding(156488, 64)
  (item_embedding): Embedding(352858, 64)
  (loss): BPRLoss()
  (user_alpha): Embedding(156488, 1)
)

## 5. Pick a sample user

Select a single user from the test set who has at least 2 test interactions.

In [27]:
uid_field = dataset.uid_field
iid_field = dataset.iid_field

test_uids = test_data.dataset.inter_feat[uid_field].numpy()
train_uids = train_data.dataset.inter_feat[train_data.dataset.uid_field].numpy()
train_iids = train_data.dataset.inter_feat[train_data.dataset.iid_field].numpy()

eligible = [u for u in np.unique(test_uids) if int((test_uids == u).sum()) >= 2]
rng = np.random.RandomState(666)
chosen_uid = rng.choice(eligible)

n_train = int((train_uids == chosen_uid).sum())
n_test = int((test_uids == chosen_uid).sum())
token = dataset.id2token(uid_field, chosen_uid)
print(f"User: internal_id={chosen_uid}, token={token}, train={n_train}, test={n_test}")

User: internal_id=130952, token=1097365, train=36, test=6


## 6. Generate recommendations + show interaction history

Display the user's training history, top‑K recommendations, and held-out test items.

In [32]:
title_id2token = {v: k for k, v in dataset.field2token_id["title"].items()}
store_id2token = {v: k for k, v in dataset.field2token_id["store"].items()}
item_prices = dataset.item_feat["price"].numpy()

def _item_row(item_id: int) -> dict:
    item_token = dataset.id2token(iid_field, item_id)
    title = title_id2token.get(int(dataset.item_feat["title"][item_id].item()), "")
    return {"Item ID": item_token, "Title": title}

uid = chosen_uid
print(f"\n{'=' * 80}")
print(f"USER (internal_id={uid}, token={dataset.id2token(uid_field, uid)})")
print(f"  {n_train} train interactions, {n_test} test interactions\n")

# --- Interaction history (train items) ---
history_iids = train_iids[train_uids == uid]
history_rows = [_item_row(int(iid)) for iid in history_iids]
display(pd.DataFrame(history_rows).style.set_caption("Interaction history (training items)"))

# --- Top-K recommendations ---
user_tensor = torch.tensor([uid], device=DEVICE)
with torch.no_grad():
    scores = model.full_sort_predict({config["USER_ID_FIELD"]: user_tensor})

topk_scores, topk_indices = torch.topk(scores, k=TOPK)

rec_rows = []
for rank, (item_id, score) in enumerate(zip(topk_indices.cpu().numpy(), topk_scores.cpu().numpy()), 1):
    row = _item_row(int(item_id))
    row["Rank"] = rank
    row["Score"] = f"{score:.4f}"
    rec_rows.append(row)

rec_df = pd.DataFrame(rec_rows)
display(rec_df.style.set_caption(f"Top-{TOPK} recommendations"))

# --- Held-out test items ---
test_iids = test_data.dataset.inter_feat[iid_field].numpy()
test_uid_array = test_data.dataset.inter_feat[uid_field].numpy()
truth_iids = test_iids[test_uid_array == uid]
truth_rows = [_item_row(int(iid)) for iid in truth_iids]
display(pd.DataFrame(truth_rows).style.set_caption("Held-out test items"))


USER (internal_id=130952, token=1097365)
  36 train interactions, 6 test interactions



,Item ID,Title
0,200564,MaveUp Eyelash Growth Serum Booster. Enhanced Length. Works Like Magic. Fuller Lashes & Mesmerizing Look. Nourishing. Thicker. Longer. Your Secret to Gorgeous Eyes. Alluring. Must Have.
1,6556,Silicon mix hair treatment and shampoo 16 ounce
2,270322,Beverlee 26 Inch Black Kinky Headband Wig Long Yaki Straight Headband Wigs for Black Women Easy To Wear Wig Synthetic None Lace Front Headband Wig
3,328423,"Red Wigs for Women,Long Curly Wavy Burgundy wig Synthetic Heat Resistant Ombre Red with Dark Roots Natural Middle Part Hair Wig for Daily Party Cosplay 24inch"
4,277198,"WANMEI 4PCS Afro Twist Comb Set - Includes 1 Big Holes Sponge Brushes+ 1 Twist Comb+1 Metal Styling Comb+1 Plastic Styling Comb - Afro Hair Combs For Barber, Salon And Home Personal Use"
5,275774,"Claw Clips for Thick Hair Large Claw Clips CEELGON Hair Clips for Women 4.3 Inches Pack of 4 (Gold,Brown)"
6,116963,SIMIYA Cotton Underwear for Women 7 Pack Bikini Panties Breathable Ladies Underwear Invisible Hipster Panties Women briefs
7,279584,WEESO Womens Summer Tank Tops Spaghetti Strap Camisole Sleeveless Racerback Tops
8,265623,Euaoxnc Women Lace Crochet Hollow Out Tank Tops Casual Blouse Summer Halter Neck Sleeveless Cocktail Shirts Clubwear
9,261587,"Estarer Gym Bag for Men & Women, 3 In 1 Sport Duffels Bag/Backpack Water-resistant, Travel Duffel Bag With Wet Pocket/Shoes Compartment (Black)"


,Item ID,Title,Rank,Score
0,249566,Amella Hair Brazilian Body Wave 3 Bundles with Free Part Lace Closure (16 18 20 with 12) 8A Grade Unprocessed Virgin Human Hair with Closure Brazilian Body Wave Natural Black Color,1,6.0107
1,231574,SUYYA Tape in Hair Extensions Human Hair 100% Remy Human Hair Darkest Brown 50g/pack 20pcs Straight Seamless Skin Weft Tape in Human Hair Extensions(24 inches #2 Darkest Brown),2,5.6316
2,166455,DOORES Hair Extensions Wire Hair Extensions Balayage Dark Brown to Chestnut Brown 20 Inch 110g Natural Hair Extensions Wire Hair Extensions with Invisible Fish Line Straight,3,5.2462
3,321918,"Abbily Hair 10Inch Brazilian Deep Wave Bundles 10A Brazolian Virgin Human Hair Bundles Natural Black Color(10Inch, Deep Bundles)",4,5.2116
4,277302,Straight 3 Bundles Human Hair 100% Unprocessed Remy Brazilian Virgin Hair (18 20 22 Inch) Double Weft Weave Bundles Human Hair Black Hair Bundles Extension Natural Color,5,5.2088
5,255524,Amella Hair Brazilian Straight Human Hair 4 Bundles 16 18 20 22inch 10A Unprocessed Virgin Straight Human Hair Bundles Natural Black Color,6,5.0721
6,351267,Vigorous Black Wig with Bangs Synthetic Long Black Wigs for Women Natural Wigs with Bangs,7,4.8061
7,275449,Straight Bundles with Closure Human Hair 24 26 28+20 Brazilian Human Hair Bundles with Closure Straight 100% Unprocessed Virgin Weave Hair Bundles with Closure 10A Bundles Human Hair with Lace Closure,8,4.7898
8,266553,"Amella Hair 8A Unprocessed Brazilian Human Hair Bundles With Frontal Closure (16 18 20+16Frontal, Natural Black) Brazilian Body Wave With 13x4 Lace Frontal Closure Pre Plucked",9,4.7654
9,249381,GOO GOO Seamless Clip In Hair Extensions Remy Real Human Hair Extension with Invisible PU Skin Weft 20 Inch 150g 7pcs Dark Brown Natural & Thick & Straight Hair Extensions for Women,10,4.7403


,Item ID,Title
0,278651,"Promax Hair Cutting Scissors,6.5 Inch Hairdressing Scissor, Premium Stainless Steel Razor with Sharp Edge Blade & Salon Scissors, for Men, Women, Barber, Kids, Adults, Pets - 210-10240R"
1,281858,"IPL Laser Permanent Hair Removal Device for Women and Men, Painless Best Hair Remover Whole Body Facial Armpits Back Legs Arms Face Bikini Line, 999,999 Flashes, Corded at-Home"
2,333141,Aveiyce Highlight Ombre HD Transparent 13x4 Lace Front Wigs Human Hair 180% Density Honey Blonde 4/27 Deep Wave Lace Front Wigs Human Hair for Black Women Frontal Curly Wigs Pre Plucked with Baby Hair 24 Inch
3,270194,"BKTD Wavy Pink Wig With Bangs For Women, Short Pastel Curly Bob Colorful Purple Pink Wigs, Shoulder Length Synthetic Loose Wave Colored Wig For Girls Party Costume Cosplay Daily Use (14 Pink)"
4,278784,ORSUNCER Long Ombre Brown Layered Wigs for Women Middle Part Brown Wig with Dark Roots Synthetic Hair Wigs for Daily Party Wig
5,249566,Amella Hair Brazilian Body Wave 3 Bundles with Free Part Lace Closure (16 18 20 with 12) 8A Grade Unprocessed Virgin Human Hair with Closure Brazilian Body Wave Natural Black Color
